# B1 exploration · What actually goes into a model call?

Start with the [six B1 snippets](../snippets/b1/README.md). This notebook opens
up their smaller mechanisms: message roles, token boundaries, response fields,
conversation history, output limits and measurements.

Use **predict → run → inspect → change one thing**. Code is complete, so you
can focus on experiments before attempting the [B1 application lab](../labs/container_help_desk/B1.md).

Default Run All performs the deterministic experiments and labels model calls
SKIPPED. Set `RUN_LIVE = True` in setup for real inference. No canned model
answers are substituted. Prepare dependencies using [the notebook guide](README.md).
The tokenizer may need internet once to cache its vocabulary.

Restart the kernel and Run All after changing setup. Running cells out of order
can retain old variables; a variable from yesterday is not a new observation.

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

TRACK_B = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'snippets/b1').is_dir()), None)
if TRACK_B is None:
    raise RuntimeError('Open this notebook from track-b or track-b/notebooks.')
for env_file in [TRACK_B / 'notebooks/.env', TRACK_B / 'snippets/.env',
                 TRACK_B / '.env', TRACK_B.parent / '.env']:
    load_dotenv(env_file)

RUN_LIVE = False  # Change to True, then restart and Run All for actual model experiments.
MODEL = 'gpt-4.1-mini'
if RUN_LIVE and not os.environ.get('OPEN_AI_KEY'):
    raise ValueError('Configure OPEN_AI_KEY before enabling live calls.')
print('Live inference enabled:', RUN_LIVE)

## 1. A word is not a token

**Predict:** will adding one space, changing case or using a container ID add
exactly one token? Tokenization uses a learned vocabulary; it is not a word counter.
This is the selected model's tokenizer, not a universal tokenizer for every model.

In [ ]:
import tiktoken
encoder = tiktoken.encoding_for_model(MODEL)
samples = ['container', ' container', 'Container', 'MSKU1234567', 'port 🚢']
for text in samples:
    token_ids = encoder.encode(text)
    print(repr(text), '->', token_ids, 'count:', len(token_ids))
    assert encoder.decode(token_ids) == text

**Inspect:** compare actual counts, not assumed word boundaries. A leading space
can change which vocabulary item is used. The assertion establishes that the
whole token sequence decodes to the input; it does not say every token is a word.

**Change:** add punctuation or replace the ID with one from PortOps. Explain
which token boundaries moved before drawing a conclusion about input cost.

## 2. One token may contain only part of a character

**Predict:** can every token in a ship emoji be decoded separately as a clean
Unicode character? Token boundaries can divide UTF-8 bytes.

In [ ]:
text = 'port 🚢'
token_ids = encoder.encode(text)
for token_id in token_ids:
    print(token_id, encoder.decode_single_token_bytes(token_id))
joined_bytes = b''.join(encoder.decode_single_token_bytes(t) for t in token_ids)
print('Joined bytes:', joined_bytes)
print('Decoded together:', joined_bytes.decode('utf-8'))

**Inspect:** bytes are the evidence here. Decoding a single incomplete byte
sequence can produce a replacement character even though the complete sequence
is valid. That does not mean the original text was corrupted.

**Change:** try an accented word or another emoji. This is why arbitrary token
slices need care when displayed as independent strings.

## 3. Role and content carry different information

**Predict:** does `Answer in two short sentences` introduce any fact about a
particular container? It asks for behavior. It does not connect an operational source.

In [ ]:
question = 'What is container dwell time in shipping?'
messages = [
    {'role': 'system', 'content': 'Explain shipping terms for a new colleague in two short sentences.'},
    {'role': 'user', 'content': question},
]
print(json.dumps(messages, indent=2))
print('Content-only tokens:', sum(len(encoder.encode(m['content'])) for m in messages))

## 4. Inspect the actual response, not only its generated sentence

The short snippet prints the reply. Here, keep the response so that text,
finish reason and usage can be inspected separately. This performs one real
request only when live inference is enabled.

In [ ]:
from litellm import completion, completion_cost
from time import perf_counter

response = None
if RUN_LIVE:
    started = perf_counter()
    response = completion(model='openai/' + MODEL, api_key=os.environ['OPEN_AI_KEY'],
                          messages=messages, max_tokens=120, temperature=0)
    elapsed = perf_counter() - started
    print('Reply:', response.choices[0].message.content)
    print('Finish reason:', response.choices[0].finish_reason)
    print('Usage:', response.usage.model_dump())
    print('Seconds:', round(elapsed, 3))
else:
    print('SKIPPED: real model request; set RUN_LIVE = True to run it.')

**Inspect:** a returned text field does not establish that the task was completed
correctly. Usage belongs to the request; the extracted sentence is only part
of the response.

Our earlier token count encoded content strings only. It did not reproduce the
full formatted conversation, role boundaries or every provider-side addition.
Compare it with the reported request usage without assuming they must be equal.

In [ ]:
if response is not None:
    content_tokens = sum(len(encoder.encode(m['content'])) for m in messages)
    print('Our content-only count:', content_tokens)
    print('Reported request input:', response.usage.prompt_tokens)
    print('Returned text tokens:', len(encoder.encode(response.choices[0].message.content or '')))
    print('Reported output tokens:', response.usage.completion_tokens)
    print('Estimated USD:', completion_cost(completion_response=response, model='openai/' + MODEL))
else:
    print('SKIPPED: no real response is available for usage/cost inspection.')

## 5. A follow-up needs the earlier exchange

The chosen port name below is **user-supplied fictional conversation context**,
not a lookup of an actual company record.

**Predict:** which request contains evidence for the chosen name? The Python
variable `first` is not automatically available to a later model call.

In [ ]:
first_messages = [{'role': 'user', 'content': "In our training example, call the port 'Harbor Orion'. Acknowledge briefly."}]
follow_up = {'role': 'user', 'content': 'What name did I choose for our example port?'}
first = None
history = None
if RUN_LIVE:
    first = completion(model='openai/' + MODEL, api_key=os.environ['OPEN_AI_KEY'],
                       messages=first_messages, max_tokens=60, temperature=0)
    history = first_messages + [
        {'role': 'assistant', 'content': first.choices[0].message.content}, follow_up,
    ]
    print('Second request with history:', json.dumps(history, indent=2))
else:
    print('SKIPPED: the history experiment needs the actual first reply.')

In [ ]:
if history is not None:
    for label, request in [('with history', history), ('without history', [follow_up])]:
        answer = completion(model='openai/' + MODEL, api_key=os.environ['OPEN_AI_KEY'],
                            messages=request, max_tokens=60, temperature=0)
        print(label, '->', answer.choices[0].message.content)
        print('Input tokens:', answer.usage.prompt_tokens)
else:
    print('SKIPPED: no live history to compare.')

**Inspect:** the history includes the actual first answer as well as the user's
chosen name. The isolated request contains neither. A lucky guess in the
isolated call would not establish access to the missing exchange.

**Change:** remove just the first user message while retaining the actual
assistant reply. Does that reply repeat the name? Inspect what evidence still
remains before deciding that the model somehow remembered a discarded message.

## 6. An output limit is a cap, not a request for a complete explanation

**Predict:** can an output stop because of a budget while still producing a
nonempty text field? Keep the task fixed and deliberately make the cap tiny.

In [ ]:
if RUN_LIVE:
    short = completion(
        model='openai/' + MODEL, api_key=os.environ['OPEN_AI_KEY'],
        messages=[{'role': 'user', 'content': 'Explain container dwell time in five numbered points.'}],
        max_tokens=3, temperature=0,
    )
    print('Text:', repr(short.choices[0].message.content))
    print('Finish reason:', short.choices[0].finish_reason)
    print('Output tokens:', short.usage.completion_tokens)
else:
    print('SKIPPED: output-limit experiment needs a real model call.')

**Inspect:** check the actual finish reason and the requested task. A nonempty
string alone is an inadequate completion test. The limit is a maximum, not a
guaranteed token count. Raising it can permit a longer answer without making
that answer more accurate.

## 7. Repetition is a measurement, not a guarantee of variability

**Predict:** with temperature zero, must every hosted call return exactly the
same text and take the same time? Repeat a fixed request and observe it.
Two matching outputs cannot establish a general guarantee of determinism.

In [ ]:
measurements = []
if RUN_LIVE:
    for attempt in range(2):
        started = perf_counter()
        repeated = completion(model='openai/' + MODEL, api_key=os.environ['OPEN_AI_KEY'],
                              messages=messages, max_tokens=120, temperature=0)
        measurements.append({
            'attempt': attempt + 1, 'reply': repeated.choices[0].message.content,
            'input': repeated.usage.prompt_tokens, 'output': repeated.usage.completion_tokens,
            'seconds': round(perf_counter() - started, 3),
            'estimated_usd': completion_cost(completion_response=repeated, model='openai/' + MODEL),
        })
    for row in measurements:
        print(row)
else:
    print('SKIPPED: repeated real-call measurements.')

**Inspect:** these are whole-call, non-streaming elapsed times. They include
more than generation speed. Compare useful answer quality before calling one
run “better.” This tiny experiment is not a model benchmark.

**Change:** vary only the requested output length. Explain whether a longer
reply changed billed output, waiting time and usefulness.

## 8. Optional · Keep model access separate from model capability

The same LiteLLM interface can reach the prepared Ollama model. This cell is
independently opt-in; default Run All does not require Ollama. See the
[local-model snippet setup](../snippets/b1/README.md) for the download.

In [ ]:
RUN_LOCAL = False
if RUN_LOCAL:
    local = completion(
        model='ollama_chat/hf.co/unsloth/SmolLM2-135M-Instruct-GGUF:Q4_K_M',
        api_base='http://localhost:11434', messages=messages, max_tokens=120, temperature=0,
    )
    print(local.choices[0].message.content)
    print('Local runtime usage:', local.usage)
else:
    print('SKIPPED: optional local inference; enable only after preparing Ollama.')

**Inspect:** the small local model can misinterpret the domain. A common API
shape does not establish equal capability, identical tokenization, identical
billing or equivalent answer quality.

## Explain what you observed

1. Why is a token count different from a word count?
2. Which exact request supplied the fictional port name?
3. Why can a truncated answer have a valid response object?
4. Which measurements help you choose a sufficient model for a feature?

Use these observations in the [B1 TODO lab](../labs/container_help_desk/B1.md).
Implementation references: [LiteLLM usage and cost](https://docs.litellm.ai/docs/completion/token_usage)
and [tiktoken](https://github.com/openai/tiktoken).